# 05 · Biosynthetic pathway and the activation clusters

NPClassifier pathway from COCONUT, joined on InChIKey, so only natural products have one.
Polyketides and carbohydrates are left out: with all seven classes too many expected counts
fall below 5 for the chi-square test. The control clusters the same odorants on ten
standardised descriptors, with no receptors involved. Both are read against a floor from
2,000 label permutations.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path("../..").resolve()
sys.path.append(str(REPO / "Downstream_Analysis/scripts"))
import activation_code as ac

PROBABILITIES = REPO / "Downstream_Analysis/predictions/aggregated/Reference_Tree_Predictions/reference_median_probability_wide.csv"
M2OR_PAIRS = REPO / "Model_Inputs/Data_Preparation/processed/m2or_pairs_model.csv"
ODOURS = REPO / "Chemicals/processed_data/m2or_inchi_odors_smiles.csv"
FASTA = REPO / "Phylogenetic_Analysis/data/ReferenceTree/PF13853.9606_7955_7740_7764_75743_137246.fa"
CLASSES = REPO / "Phylogenetic_Analysis/data/HumanTree/human433_OR_classes.csv"
TABLES = Path("results")
TABLES.mkdir(exist_ok=True)
PATHWAYS = REPO / "Chemicals/processed_data/ligand_pathways.csv"

In [2]:
hybrid_table = pd.read_csv(TABLES / "hybrid_matrix.csv", index_col=0)
receptor_ids = list(hybrid_table.index)
odorant_ids = list(hybrid_table.columns)
hybrid = hybrid_table.values
clusters = pd.read_csv(TABLES / "odorant_clusters.csv")
info, display_order = ac.clusters_from_table(clusters)
clustered_ids = clusters.smiles_id.values
labels = clusters.cluster.values
smiles_by_odorant, tags_by_odorant = ac.odour_tags(M2OR_PAIRS, ODOURS, odorant_ids)

In [3]:
pathway_result = ac.pathway_association(clustered_ids, labels, smiles_by_odorant, ODOURS, PATHWAYS)
pathway_result["recovery"].to_csv(TABLES / "pathway_association.csv", index=False)

contingency = pathway_result["results"]["receptor clusters"]["ct"]
contingency.index = [dict(zip(clusters.cluster, clusters.cluster_tag))[c] for c in contingency.index]
contingency.sort_index().to_csv(TABLES / "pathway_by_cluster.csv")

print(f"clustered odorants with an analysed pathway: {int(pathway_result['use'].sum())}")
pathway_result["recovery"]

clustered odorants with an analysed pathway: 427


,clustering,cramers_v,chi2_p,floor,floor_95th,n
0,receptor clusters,0.355935,6.276079e-35,0.106909,0.136658,427
1,descriptor clusters,0.569120,2.003573e-104,0.107584,0.138137,427
